In [1]:
print("Name: K ADHITHYA")
print("Roll No: 24BAD004")

Name: K ADHITHYA
Roll No: 24BAD004


In [2]:

# installing the libraries
!pip install -q transformers streamlit accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 109.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 117.5 MB/s eta 0:00:00


## Part A: Environment setup and model selection

In [3]:
%%writefile llm.py
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# small model so it runs on colab easily
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"


def load_model():
    # load tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype="auto")
    model.eval()
    return tokenizer, model


def generate_response(message, tokenizer, model, history=None):
    # handle empty input
    if not message or not message.strip():
        return "Please enter a message."

    try:
        # make the list of messages (system + old chat + new message)
        messages = [{"role": "system", "content": "You are a helpful assistant."}]
        if history:
            messages = messages + history
        messages.append({"role": "user", "content": message})

        # convert to the format the model needs
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        # generate the reply
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
            )

        # remove the prompt part and decode only the new tokens
        new_tokens = output[0][inputs["input_ids"].shape[1]:]
        return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    except Exception as e:
        return "Error: " + str(e)

Writing llm.py


In [4]:
from llm import load_model, generate_response, MODEL_NAME

print("Loading model:", MODEL_NAME)
tokenizer, model = load_model()
print("Model loaded on:", model.device)

# checking what the tokenizer does
print("Tokens:", tokenizer("Hello world")["input_ids"])

# one simple test prompt
print("Reply:", generate_response("Say hello in one sentence.", tokenizer, model))

Loading model: Qwen/Qwen2.5-0.5B-Instruct


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded on: cpu
Tokens: [9707, 1879]
Reply: Hello, how can I help you today?


## Part B: Prompt handling and response generation

In [5]:
# testing with different prompts (first one is empty to check error handling)
prompts = [
    "",
    "What is the capital of France?",
    "Hi! How are you today?",
    "Write a Python function to reverse a string.",
    "Explain overfitting in two sentences.",
    "Give me three tips to prepare for exams.",
]

for i, p in enumerate(prompts, 1):
    print("\n[" + str(i) + "] User:", repr(p))
    print("Bot:", generate_response(p, tokenizer, model))


[1] User: ''
Bot: Please enter a message.

[2] User: 'What is the capital of France?'
Bot: The capital of France is Paris.

[3] User: 'Hi! How are you today?'
Bot: I'm doing well, thank you for asking! How about you?

[4] User: 'Write a Python function to reverse a string.'
Bot: Certainly! Here's an example of a Python function that reverses a string:

```python
def reverse_string(string):
    """
    Reverses the given string.
    
    Parameters:
    - string (str): The input string to be reversed.

    Returns:
    - str: The reversed string.
    """
    return string[::-1]
```

This function takes a single argument, `string`, which is the input string you want to reverse. It uses slicing with a step size of `-1` (`[::-1]`) to reverse the order of characters in the string and returns the reversed string as a new value.

Here's an example usage of this function:

```python
my_string = "hello"
reversed_string = reverse_string(my_string)
print(reversed_string)  # Output: "olleh"
```



## Part C: Chatbot interface using Streamlit

In [6]:
%%writefile app.py
import streamlit as st
from llm import load_model, generate_response

st.set_page_config(page_title="LLM Chatbot")
st.title("LLM Chatbot")


# load the model only once
@st.cache_resource
def get_model():
    return load_model()


tokenizer, model = get_model()

# keep chat history in session state
if "history" not in st.session_state:
    st.session_state.history = []

# clear chat button
if st.sidebar.button("Clear chat"):
    st.session_state.history = []
    st.rerun()

# show old messages
for m in st.session_state.history:
    with st.chat_message(m["role"]):
        st.write(m["content"])

# chat input box
user_msg = st.chat_input("Type your message...")

if user_msg:
    with st.chat_message("user"):
        st.write(user_msg)

    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            reply = generate_response(user_msg, tokenizer, model, st.session_state.history)
        st.write(reply)

    # save both messages in history
    st.session_state.history.append({"role": "user", "content": user_msg})
    st.session_state.history.append({"role": "assistant", "content": reply})

Writing app.py


In [7]:
# stop old streamlit (if any) and start the app in background
!pkill -f streamlit
!nohup streamlit run app.py --server.port 8501 --server.enableCORS false --server.enableXsrfProtection false > /content/logs.txt 2>&1 &

In [ ]:
# public link using cloudflare, open the trycloudflare.com link that is printed
# keep this cell running while using the app
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!./cloudflared tunnel --url http://localhost:8501

2026-10-12T12:55:00Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-10-12T12:55:00Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-10-12T12:55:07Z INF +--------------------------------------------------------------------------------------------+
2026-10-12T12:55:07Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-10-12T12:55:07Z INF |  https://refresh-continuously-does-seasons.trycloudfla